# 1. Import Libraries

In [47]:
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# 2. Define Paths

In [25]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "lending_club_resolved_full.csv"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Input path:", INPUT_PATH)
print("File exists:", INPUT_PATH.exists())

if not INPUT_PATH.exists():
    raise FileNotFoundError("Run Notebook 01 first. Expected file: data/processed/lending_club_resolved_full.csv")

Project root: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator
Input path: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\lending_club_resolved_full.csv
File exists: True


# 3. Load Dataset

In [26]:
df = pd.read_csv(INPUT_PATH)

print("Dataset loaded successfully.", df.shape)

df.head()

Dataset loaded successfully. (396030, 28)


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,title,dti,earliest_cr_line,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,address,default_flag
0,"10,000.0000",36 months,11.4400,329.4800,B,B4,Marketing,10+ years,RENT,"117,000.0000",Not Verified,Jan-2015,Fully Paid,vacation,Vacation,26.2400,Jun-1990,16.0000,0.0000,"36,369.0000",41.8000,25.0000,w,INDIVIDUAL,0.0000,0.0000,"0174 Michelle Gateway\r\nMendozaberg, OK 22690",0
1,"8,000.0000",36 months,11.9900,265.6800,B,B5,Credit analyst,4 years,MORTGAGE,"65,000.0000",Not Verified,Jan-2015,Fully Paid,debt_consolidation,Debt consolidation,22.0500,Jul-2004,17.0000,0.0000,"20,131.0000",53.3000,27.0000,f,INDIVIDUAL,3.0000,0.0000,"1076 Carney Fort Apt. 347\r\nLoganmouth, SD 05113",0
2,"15,600.0000",36 months,10.4900,506.9700,B,B3,Statistician,< 1 year,RENT,"43,057.0000",Source Verified,Jan-2015,Fully Paid,credit_card,Credit card refinancing,12.7900,Aug-2007,13.0000,0.0000,"11,987.0000",92.2000,26.0000,f,INDIVIDUAL,0.0000,0.0000,"87025 Mark Dale Apt. 269\r\nNew Sabrina, WV 05113",0
3,"7,200.0000",36 months,6.4900,220.6500,A,A2,Client Advocate,6 years,RENT,"54,000.0000",Not Verified,Nov-2014,Fully Paid,credit_card,Credit card refinancing,2.6000,Sep-2006,6.0000,0.0000,"5,472.0000",21.5000,13.0000,f,INDIVIDUAL,0.0000,0.0000,"823 Reid Ford\r\nDelacruzside, MA 00813",0
4,"24,375.0000",60 months,17.2700,609.3300,C,C5,Destiny Management Inc.,9 years,MORTGAGE,"55,000.0000",Verified,Apr-2013,Charged Off,credit_card,Credit Card Refinance,33.9500,Mar-1999,13.0000,0.0000,"24,584.0000",69.8000,43.0000,f,INDIVIDUAL,1.0000,0.0000,"679 Luna Roads\r\nGreggshire, VA 11650",1


# 4. Data Preprocessing

In [27]:
# Confirm target variable

if "default_flag" not in df.columns:
    if "loan_status" not in df.columns:
        raise ValueError("Neither default_flag nor loan_status exists.")
    
    df["default_flag"] = np.where(df["loan_status"].eq("Charged Off"), 1, 0)

target_summary = (
    df["default_flag"]
    .value_counts()
    .sort_index()
    .to_frame("count")
    .assign(
        label=["Fully Paid / Non-default", "Charged Off / Default"],
        pct=lambda x: x["count"] / x["count"].sum() * 100
    )
)

display(target_summary)

,count,label,pct
default_flag,,,
0,318357,Fully Paid / Non-default,80.3871
1,77673,Charged Off / Default,19.6129


In [28]:
# Helper function

def parse_term_months(x):
    """
    Convert term string such as '36 months' into numeric 36.
    """
    if pd.isna(x):
        return np.nan
    return float(str(x).strip().split()[0])


def parse_emp_length(x):
    """
    Convert employment length into numeric years.
    10+ years -> 10
    < 1 year  -> 0
    1 year    -> 1
    2 years   -> 2
    Missing   -> NaN
    """
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    
    if x == "10+ years":
        return 10.0
    elif x == "< 1 year":
        return 0.0
    elif x == "1 year":
        return 1.0
    elif "years" in x:
        return float(x.replace(" years", ""))
    else:
        return np.nan


def parse_date_month_year(x):
    """
    Parse Lending Club month-year strings such as 'Jan-2015'.
    Returns pandas datetime.
    """
    return pd.to_datetime(x, format="%b-%Y", errors="coerce")


def cap_series_at_quantile(series, upper_q=0.99):
    """
    Cap upper tail at selected quantile.
    Does not cap lower tail unless values are invalid elsewhere.
    """
    cap_value = series.quantile(upper_q)
    return series.clip(upper=cap_value), cap_value


def safe_log1p(series):
    """
    Apply log1p after forcing negative values to NaN.
    """
    s = series.copy()
    s = s.where(s >= 0, np.nan)
    return np.log1p(s)

In [29]:
# Clean working dataframe

clean_df = df.copy()
print("Initial shape:", clean_df.shape)

Initial shape: (396030, 28)


In [30]:
# Date parsing and time features

# Parse issue date
clean_df["issue_date"] = parse_date_month_year(clean_df["issue_d"])

# Parse earliest credit line date
clean_df["earliest_cr_line_date"] = parse_date_month_year(clean_df["earliest_cr_line"])

# Extract issue year and month
clean_df["issue_year"] = clean_df["issue_date"].dt.year
clean_df["issue_month"] = clean_df["issue_date"].dt.month

# Credit history length in years
clean_df["credit_history_length"] = (
    (clean_df["issue_date"] - clean_df["earliest_cr_line_date"]).dt.days / 365.25
)

date_check = clean_df[[
    "issue_d",
    "issue_date",
    "earliest_cr_line",
    "earliest_cr_line_date",
    "issue_year",
    "issue_month",
    "credit_history_length"
]].head(10)

display(date_check)

print("Missing issue_date:", clean_df["issue_date"].isna().sum())
print("Missing earliest_cr_line_date:", clean_df["earliest_cr_line_date"].isna().sum())
print("Missing credit_history_length:", clean_df["credit_history_length"].isna().sum())

,issue_d,issue_date,earliest_cr_line,earliest_cr_line_date,issue_year,issue_month,credit_history_length
0,Jan-2015,2015-01-01,Jun-1990,1990-06-01,2015,1,24.5859
1,Jan-2015,2015-01-01,Jul-2004,2004-07-01,2015,1,10.5024
2,Jan-2015,2015-01-01,Aug-2007,2007-08-01,2015,1,7.4196
3,Nov-2014,2014-11-01,Sep-2006,2006-09-01,2014,11,8.1670
4,Apr-2013,2013-04-01,Mar-1999,1999-03-01,2013,4,14.0862
5,Sep-2015,2015-09-01,Jan-2005,2005-01-01,2015,9,10.6639
6,Sep-2015,2015-09-01,Aug-2005,2005-08-01,2015,9,10.0835
7,Sep-2012,2012-09-01,Sep-1994,1994-09-01,2012,9,18.0014
8,Oct-2014,2014-10-01,Jun-1994,1994-06-01,2014,10,20.3340
9,Apr-2012,2012-04-01,Dec-1997,1997-12-01,2012,4,14.3326


Missing issue_date: 0
Missing earliest_cr_line_date: 0
Missing credit_history_length: 0


In [31]:
# Term and employment features

clean_df["term_months"] = clean_df["term"].apply(parse_term_months)

clean_df["emp_length_num"] = clean_df["emp_length"].apply(parse_emp_length)

# Missing indicator
clean_df["emp_length_missing"] = clean_df["emp_length_num"].isna().astype(int)

display(
    clean_df[[
        "term", "term_months",
        "emp_length", "emp_length_num", "emp_length_missing"
    ]].head(20)
)

summary = clean_df[["term_months", "emp_length_num", "emp_length_missing"]].describe().T
display(summary)

print("term_months missing:", clean_df["term_months"].isna().sum())
print("emp_length_num missing:", clean_df["emp_length_num"].isna().sum())

,term,term_months,emp_length,emp_length_num,emp_length_missing
0,36 months,36.0000,10+ years,10.0000,0
1,36 months,36.0000,4 years,4.0000,0
2,36 months,36.0000,< 1 year,0.0000,0
3,36 months,36.0000,6 years,6.0000,0
4,60 months,60.0000,9 years,9.0000,0
5,36 months,36.0000,10+ years,10.0000,0
6,36 months,36.0000,2 years,2.0000,0
7,36 months,36.0000,10+ years,10.0000,0
8,60 months,60.0000,10+ years,10.0000,0
9,36 months,36.0000,3 years,3.0000,0


,count,mean,std,min,25%,50%,75%,max
term_months,"396,030.0000",41.6981,10.2120,36.0000,36.0000,36.0000,36.0000,60.0000
emp_length_num,"377,729.0000",5.9386,3.6456,0.0000,3.0000,6.0000,10.0000,10.0000
emp_length_missing,"396,030.0000",0.0462,0.2099,0.0000,0.0000,0.0000,0.0000,1.0000


term_months missing: 0
emp_length_num missing: 18301


In [32]:
# Income, balance, and affordability features

# Log income
clean_df["annual_inc_log"] = safe_log1p(clean_df["annual_inc"])

# Monthly income
clean_df["monthly_income"] = clean_df["annual_inc"] / 12

# Installment to income ratio
clean_df["installment_to_income"] = np.where(
    clean_df["monthly_income"] > 0,
    clean_df["installment"] / clean_df["monthly_income"],
    np.nan
)

# Log revolving balance
clean_df["revol_bal_log"] = safe_log1p(clean_df["revol_bal"])

feature_preview_cols = [
    "annual_inc",
    "annual_inc_log",
    "monthly_income",
    "installment",
    "installment_to_income",
    "revol_bal",
    "revol_bal_log"
]

clean_df[feature_preview_cols].head(20)
clean_df[feature_preview_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
annual_inc,"396,030.0000","74,203.1758","61,637.6212",0.0000,"19,000.0000","28,000.0000","45,000.0000","64,000.0000","90,000.0000","150,000.0000","250,000.0000","8,706,582.0000"
annual_inc_log,"396,030.0000",11.0671,0.5246,0.0000,9.8522,10.2400,10.7144,11.0667,11.4076,11.9184,12.4292,15.9796
monthly_income,"396,030.0000","6,183.5980","5,136.4684",0.0000,"1,583.3333","2,333.3333","3,750.0000","5,333.3333","7,500.0000","12,500.0000","20,833.3333","725,548.5000"
installment,"396,030.0000",431.8497,250.7278,16.0800,54.4300,109.5100,250.3300,375.4300,567.3000,925.6000,"1,202.3730","1,533.8100"
installment_to_income,"396,029.0000",0.0792,0.0479,0.0001,0.0106,0.0213,0.0476,0.0736,0.1061,0.1542,0.1857,15.2998
revol_bal,"396,030.0000","15,844.5399","20,591.8361",0.0000,154.0000,"1,685.0000","6,025.0000","11,181.0000","19,620.0000","41,066.5500","86,039.6200","1,743,266.0000"
revol_bal_log,"396,030.0000",9.1872,1.2269,0.0000,5.0434,7.4301,8.7038,9.3221,9.8844,10.6230,11.3626,14.3713


In [33]:
# Missing indicators and numeric imputation-ready features
clean_df["mort_acc_missing"] = clean_df["mort_acc"].isna().astype(int)
clean_df["pub_rec_bankruptcies_missing"] = clean_df["pub_rec_bankruptcies"].isna().astype(int)
clean_df["revol_util_missing"] = clean_df["revol_util"].isna().astype(int)

missing_indicator_summary = clean_df[[
    "mort_acc_missing",
    "emp_length_missing",
    "pub_rec_bankruptcies_missing",
    "revol_util_missing"
]].mean().to_frame("missing_rate")

display(missing_indicator_summary)

,missing_rate
mort_acc_missing,0.0954
emp_length_missing,0.0462
pub_rec_bankruptcies_missing,0.0014
revol_util_missing,0.0007


In [34]:
# Outlier capping
cap_columns = [
    "dti",
    "revol_util",
    "installment_to_income",
    "credit_history_length"
]

cap_values = {}

for col in cap_columns:
    if col in clean_df.columns:
        clean_df[f"{col}_capped"], cap_value = cap_series_at_quantile(clean_df[col], upper_q=0.99)
        cap_values[col] = cap_value

cap_summary = pd.DataFrame({
    "column": list(cap_values.keys()),
    "cap_99pct": list(cap_values.values())
})

display(cap_summary)

check_cols = []
for col in cap_columns:
    if f"{col}_capped" in clean_df.columns:
        check_cols.extend([col, f"{col}_capped"])

display(clean_df[check_cols].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
).T)

,column,cap_99pct
0,dti,36.4300
1,revol_util,98.0000
2,installment_to_income,0.1857
3,credit_history_length,38.4942


,count,mean,std,min,1%,5%,50%,95%,99%,max
dti,"396,030.0000",17.3795,18.0191,0.0000,1.5600,4.6800,16.9100,31.5800,36.4300,"9,999.0000"
dti_capped,"396,030.0000",17.3261,8.0685,0.0000,1.5600,4.6800,16.9100,31.5800,36.4300,36.4300
revol_util,"395,754.0000",53.7917,24.4522,0.0000,1.0000,11.2000,54.8000,92.0000,98.0000,892.3000
revol_util_capped,"395,754.0000",53.7681,24.3731,0.0000,1.0000,11.2000,54.8000,92.0000,98.0000,98.0000
installment_to_income,"396,029.0000",0.0792,0.0479,0.0001,0.0106,0.0213,0.0736,0.1542,0.1857,15.2998
installment_to_income_capped,"396,029.0000",0.0790,0.0407,0.0001,0.0106,0.0213,0.0736,0.1542,0.1857,0.1857
credit_history_length,"396,030.0000",15.7543,7.2011,2.9979,4.0000,6.1656,14.3354,29.6619,38.4942,70.1629
credit_history_length_capped,"396,030.0000",15.7101,7.0346,2.9979,4.0000,6.1656,14.3354,29.6619,38.4942,38.4942


In [35]:
# Clean categorical columns

categorical_cols_to_clean = [
    "home_ownership",
    "verification_status",
    "purpose",
    "initial_list_status",
    "application_type",
    "grade",
    "sub_grade",
    "loan_status"
]

for col in categorical_cols_to_clean:
    if col in clean_df.columns:
        clean_df[col] = (
            clean_df[col]
            .astype(str)
            .str.strip()
            .replace({"nan": np.nan})
        )

for col in categorical_cols_to_clean:
    if col in clean_df.columns:
        print("=" * 80)
        print(col)
        display(clean_df[col].value_counts(dropna=False).head(30).to_frame("count"))

home_ownership


,count
home_ownership,
MORTGAGE,198348
RENT,159790
OWN,37746
OTHER,112
NONE,31
ANY,3


verification_status


,count
verification_status,
Verified,139563
Source Verified,131385
Not Verified,125082


purpose


...

,count
grade,
B,116018
C,105987
A,64187
D,63524
E,31488
F,11772
G,3054


sub_grade


,count
sub_grade,
B3,26655
B4,25601
C1,23662
C2,22580
B2,22495
B5,22085
C3,21221
C4,20280
B1,19182


loan_status


,count
loan_status,
Fully Paid,318357
Charged Off,77673


In [36]:
# Decide primary model features

primary_numeric_features = [
    "loan_amnt",
    "term_months",
    "annual_inc_log",
    "dti_capped",
    "emp_length_num",
    "emp_length_missing",
    "open_acc",
    "pub_rec",
    "revol_bal_log",
    "revol_util_capped",
    "revol_util_missing",
    "total_acc",
    "mort_acc",
    "mort_acc_missing",
    "pub_rec_bankruptcies",
    "pub_rec_bankruptcies_missing",
    "credit_history_length_capped"
]

primary_categorical_features = [
    "home_ownership",
    "verification_status",
    "purpose",
    "initial_list_status",
    "application_type"
]

# Columns retained for pricing and benchmark analysis, not primary model input
benchmark_pricing_columns = [
    "loan_status",
    "default_flag",
    "grade",
    "sub_grade",
    "int_rate",
    "installment",
    "loan_amnt",
    "term_months",
    "issue_date"
]

# Keep only columns that exist
primary_numeric_features = [c for c in primary_numeric_features if c in clean_df.columns]
primary_categorical_features = [c for c in primary_categorical_features if c in clean_df.columns]
benchmark_pricing_columns = [c for c in benchmark_pricing_columns if c in clean_df.columns]

print("Primary numeric features:")
print(primary_numeric_features)

print("\nPrimary categorical features:")
print(primary_categorical_features)

print("\nBenchmark/pricing columns:")
print(benchmark_pricing_columns)

feature_decision = pd.DataFrame({
    "column": (
        primary_numeric_features
        + primary_categorical_features
        + benchmark_pricing_columns
        + ["emp_title", "title", "address", "issue_d", "earliest_cr_line"]
    ),
    "role": (
        ["primary_numeric_feature"] * len(primary_numeric_features)
        + ["primary_categorical_feature"] * len(primary_categorical_features)
        + ["benchmark_pricing_or_target_column"] * len(benchmark_pricing_columns)
        + ["excluded_or_raw_reference"] * 5
    )
})

display(feature_decision)

Primary numeric features:
['loan_amnt', 'term_months', 'annual_inc_log', 'dti_capped', 'emp_length_num', 'emp_length_missing', 'open_acc', 'pub_rec', 'revol_bal_log', 'revol_util_capped', 'revol_util_missing', 'total_acc', 'mort_acc', 'mort_acc_missing', 'pub_rec_bankruptcies', 'pub_rec_bankruptcies_missing', 'credit_history_length_capped']

Primary categorical features:
['home_ownership', 'verification_status', 'purpose', 'initial_list_status', 'application_type']

Benchmark/pricing columns:
['loan_status', 'default_flag', 'grade', 'sub_grade', 'int_rate', 'installment', 'loan_amnt', 'term_months', 'issue_date']


,column,role
0,loan_amnt,primary_numeric_feature
1,term_months,primary_numeric_feature
2,annual_inc_log,primary_numeric_feature
3,dti_capped,primary_numeric_feature
4,emp_length_num,primary_numeric_feature
5,emp_length_missing,primary_numeric_feature
6,open_acc,primary_numeric_feature
7,pub_rec,primary_numeric_feature
8,revol_bal_log,primary_numeric_feature
9,revol_util_capped,primary_numeric_feature


In [37]:
# Build final modeling dataframe

final_columns = (
    ["default_flag"]
    + primary_numeric_features
    + primary_categorical_features
    + [
        "loan_status",
        "grade",
        "sub_grade",
        "int_rate",
        "installment",
        "issue_date"
    ]
)

# Remove duplicates while preserving order
final_columns = list(dict.fromkeys([c for c in final_columns if c in clean_df.columns]))

modeling_df = clean_df[final_columns].copy()

print("Modeling dataframe shape:", modeling_df.shape)
display(modeling_df.head())

modeling_missing = (
    pd.DataFrame({
        "column": modeling_df.columns,
        "dtype": modeling_df.dtypes.astype(str).values,
        "missing_count": modeling_df.isna().sum().values,
        "missing_pct": modeling_df.isna().mean().values * 100
    })
    .sort_values("missing_pct", ascending=False)
)

display(modeling_missing)


Modeling dataframe shape: (396030, 29)


,default_flag,loan_amnt,term_months,annual_inc_log,dti_capped,emp_length_num,emp_length_missing,open_acc,pub_rec,revol_bal_log,revol_util_capped,revol_util_missing,total_acc,mort_acc,mort_acc_missing,pub_rec_bankruptcies,pub_rec_bankruptcies_missing,credit_history_length_capped,home_ownership,verification_status,purpose,initial_list_status,application_type,loan_status,grade,sub_grade,int_rate,installment,issue_date
0,0,"10,000.0000",36.0000,11.6699,26.2400,10.0000,0,16.0000,0.0000,10.5015,41.8000,0,25.0000,0.0000,0,0.0000,0,24.5859,RENT,Not Verified,vacation,w,INDIVIDUAL,Fully Paid,B,B4,11.4400,329.4800,2015-01-01
1,0,"8,000.0000",36.0000,11.0822,22.0500,4.0000,0,17.0000,0.0000,9.9101,53.3000,0,27.0000,3.0000,0,0.0000,0,10.5024,MORTGAGE,Not Verified,debt_consolidation,f,INDIVIDUAL,Fully Paid,B,B5,11.9900,265.6800,2015-01-01
2,0,"15,600.0000",36.0000,10.6703,12.7900,0.0000,0,13.0000,0.0000,9.3917,92.2000,0,26.0000,0.0000,0,0.0000,0,7.4196,RENT,Source Verified,credit_card,f,INDIVIDUAL,Fully Paid,B,B3,10.4900,506.9700,2015-01-01
3,0,"7,200.0000",36.0000,10.8968,2.6000,6.0000,0,6.0000,0.0000,8.6076,21.5000,0,13.0000,0.0000,0,0.0000,0,8.1670,RENT,Not Verified,credit_card,f,INDIVIDUAL,Fully Paid,A,A2,6.4900,220.6500,2014-11-01
4,1,"24,375.0000",60.0000,10.9151,33.9500,9.0000,0,13.0000,0.0000,10.1099,69.8000,0,43.0000,1.0000,0,0.0000,0,14.0862,MORTGAGE,Verified,credit_card,f,INDIVIDUAL,Charged Off,C,C5,17.2700,609.3300,2013-04-01


,column,dtype,missing_count,missing_pct
13,mort_acc,float64,37795,9.5435
5,emp_length_num,float64,18301,4.6211
15,pub_rec_bankruptcies,float64,535,0.1351
10,revol_util_capped,float64,276,0.0697
2,term_months,float64,0,0.0000
4,dti_capped,float64,0,0.0000
3,annual_inc_log,float64,0,0.0000
7,open_acc,float64,0,0.0000
6,emp_length_missing,int64,0,0.0000
8,pub_rec,float64,0,0.0000


# 5. Data Splitting

In [48]:
# Stratified random split

if "default_flag" not in modeling_df.columns:
    raise ValueError("default_flag is required for stratified split.")

# First split: train 70%, temp 30%
train_df, temp_df = train_test_split(
    modeling_df,
    test_size=0.30,
    stratify=modeling_df["default_flag"],
    random_state=42
)

# Second split: valid 15%, test 15%
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["default_flag"],
    random_state=42
)

# Reset index
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary = pd.DataFrame({
    "split": ["train", "valid", "test"],
    "split_type": ["stratified_random", "stratified_random", "stratified_random"],
    "n_rows": [len(train_df), len(valid_df), len(test_df)],
    "start_date": [
        train_df["issue_date"].min(),
        valid_df["issue_date"].min(),
        test_df["issue_date"].min()
    ],
    "end_date": [
        train_df["issue_date"].max(),
        valid_df["issue_date"].max(),
        test_df["issue_date"].max()
    ],
    "default_rate": [
        train_df["default_flag"].mean(),
        valid_df["default_flag"].mean(),
        test_df["default_flag"].mean()
    ],
    "avg_int_rate": [
        train_df["int_rate"].mean(),
        valid_df["int_rate"].mean(),
        test_df["int_rate"].mean()
    ],
    "avg_loan_amnt": [
        train_df["loan_amnt"].mean(),
        valid_df["loan_amnt"].mean(),
        test_df["loan_amnt"].mean()
    ]
})

display(split_summary)

,split,split_type,n_rows,start_date,end_date,default_rate,avg_int_rate,avg_loan_amnt
0,train,stratified_random,277221,2007-06-01,2016-12-01,0.1961,13.6349,"14,110.6595"
1,valid,stratified_random,59404,2007-07-01,2016-12-01,0.1961,13.6377,"14,125.9288"
2,test,stratified_random,59405,2007-07-01,2016-12-01,0.1961,13.6619,"14,116.9144"


In [49]:
# Check target and grade distribution by split

for split_name, split_df in {
    "train": train_df,
    "valid": valid_df,
    "test": test_df
}.items():
    print("=" * 100)
    print(split_name.upper())
    
    print("Target distribution:")
    display(
        split_df["default_flag"]
        .value_counts()
        .sort_index()
        .to_frame("count")
        .assign(pct=lambda x: x["count"] / x["count"].sum() * 100)
    )
    
    if "grade" in split_df.columns:
        print("Grade distribution:")
        display(
            split_df["grade"]
            .value_counts()
            .sort_index()
            .to_frame("count")
            .assign(pct=lambda x: x["count"] / x["count"].sum() * 100)
        )

TRAIN
Target distribution:


,count,pct
default_flag,,
0,222850,80.3871
1,54371,19.6129


Grade distribution:


,count,pct
grade,,
A,44966,16.2203
B,81270,29.3160
C,74282,26.7952
D,44324,15.9887
E,21985,7.9305
F,8256,2.9781
G,2138,0.7712


VALID
Target distribution:


...

,count,pct
grade,,
A,9659,16.2598
B,17337,29.1849
C,15876,26.7255
D,9623,16.1992
E,4711,7.9304
F,1747,2.9409
G,451,0.7592


TEST
Target distribution:


,count,pct
default_flag,,
0,47754,80.3872
1,11651,19.6128


Grade distribution:


,count,pct
grade,,
A,9562,16.0963
B,17411,29.3090
C,15829,26.6459
D,9577,16.1215
E,4792,8.0667
F,1769,2.9779
G,465,0.7828


In [50]:
# Secondary robustness split: calendar-based time split
# Purpose: temporal robustness analysis, not primary model fitting

time_df = modeling_df.sort_values("issue_date").reset_index(drop=True)

time_train_df = time_df[time_df["issue_date"] < "2015-01-01"].copy()

time_valid_df = time_df[
    (time_df["issue_date"] >= "2015-01-01") &
    (time_df["issue_date"] < "2016-01-01")
].copy()

time_test_df = time_df[time_df["issue_date"] >= "2016-01-01"].copy()

time_split_summary = pd.DataFrame({
    "split": ["time_train", "time_valid", "time_test"],
    "split_type": ["calendar_time", "calendar_time", "calendar_time"],
    "n_rows": [len(time_train_df), len(time_valid_df), len(time_test_df)],
    "start_date": [
        time_train_df["issue_date"].min(),
        time_valid_df["issue_date"].min(),
        time_test_df["issue_date"].min()
    ],
    "end_date": [
        time_train_df["issue_date"].max(),
        time_valid_df["issue_date"].max(),
        time_test_df["issue_date"].max()
    ],
    "default_rate": [
        time_train_df["default_flag"].mean(),
        time_valid_df["default_flag"].mean(),
        time_test_df["default_flag"].mean()
    ],
    "avg_int_rate": [
        time_train_df["int_rate"].mean(),
        time_valid_df["int_rate"].mean(),
        time_test_df["int_rate"].mean()
    ],
    "avg_loan_amnt": [
        time_train_df["loan_amnt"].mean(),
        time_valid_df["loan_amnt"].mean(),
        time_test_df["loan_amnt"].mean()
    ]
})

display(time_split_summary)

,split,split_type,n_rows,start_date,end_date,default_rate,avg_int_rate,avg_loan_amnt
0,time_train,calendar_time,273678,2007-06-01,2014-12-01,0.1846,13.7329,"13,767.3732"
1,time_valid,calendar_time,94264,2015-01-01,2015-12-01,0.2490,13.3268,"14,908.1412"
2,time_test,calendar_time,28088,2016-01-01,2016-12-01,0.1313,13.7776,"14,824.6555"


In [54]:
# Check target and grade distribution for time-based splits

time_splits = {
    "time_train": time_train_df,
    "time_valid": time_valid_df,
    "time_test": time_test_df
}

for split_name, split_df in time_splits.items():
    print("=" * 100)
    print(split_name.upper())
    
    print("Target distribution:")
    target_dist = (
        split_df["default_flag"]
        .value_counts()
        .sort_index()
        .to_frame("count")
        .assign(pct=lambda x: x["count"] / x["count"].sum() * 100)
    )
    display(target_dist)
    
    if "grade" in split_df.columns:
        print("Grade distribution:")
        grade_dist = (
            split_df["grade"]
            .value_counts()
            .sort_index()
            .to_frame("count")
            .assign(pct=lambda x: x["count"] / x["count"].sum() * 100)
        )
        display(grade_dist)

TIME_TRAIN
Target distribution:


,count,pct
default_flag,,
0,223166,81.5433
1,50512,18.4567


Grade distribution:


,count,pct
grade,,
A,46355,16.9378
B,85379,31.1969
C,70532,25.7719
D,42923,15.6838
E,19251,7.0342
F,7396,2.7024
G,1842,0.6731


TIME_VALID
Target distribution:


...

,count,pct
grade,,
A,13721,14.5559
B,23265,24.6807
C,27189,28.8435
D,16168,17.1518
E,9742,10.3348
F,3312,3.5135
G,867,0.9198


TIME_TEST
Target distribution:


,count,pct
default_flag,,
0,24400,86.8698
1,3688,13.1302


Grade distribution:


,count,pct
grade,,
A,4111,14.6361
B,7374,26.2532
C,8266,29.4289
D,4433,15.7825
E,2495,8.8828
F,1064,3.7881
G,345,1.2283


In [55]:
# Default rate by Lending Club grade for time-based splits

time_splits = {
    "time_train": time_train_df,
    "time_valid": time_valid_df,
    "time_test": time_test_df
}

for split_name, split_df in time_splits.items():
    print("=" * 100)
    print(f"{split_name.upper()} - DEFAULT RATE BY GRADE")
    
    grade_default = (
        split_df
        .groupby("grade")
        .agg(
            n_loans=("default_flag", "size"),
            default_rate=("default_flag", "mean"),
            avg_int_rate=("int_rate", "mean"),
            avg_loan_amnt=("loan_amnt", "mean")
        )
        .reset_index()
        .sort_values("grade")
    )
    
    grade_default["default_rate"] = grade_default["default_rate"] * 100
    
    display(grade_default)

TIME_TRAIN - DEFAULT RATE BY GRADE


,grade,n_loans,default_rate,avg_int_rate,avg_loan_amnt
0,A,46355,6.2755,7.5592,"13,002.6038"
1,B,85379,12.2852,11.6092,"12,720.4579"
2,C,70532,20.6091,14.7615,"13,584.9143"
3,D,42923,27.4585,17.7056,"14,258.9433"
4,E,19251,35.8579,20.5607,"17,257.8360"
5,F,7396,41.0357,23.4935,"18,564.1225"
6,G,1842,46.3626,25.0243,"21,331.6097"


TIME_VALID - DEFAULT RATE BY GRADE


,grade,n_loans,default_rate,avg_int_rate,avg_loan_amnt
0,A,13721,7.0184,6.9958,"14,133.3959"
1,B,23265,15.2762,10.1468,"13,517.9766"
2,C,27189,25.3227,13.3555,"14,023.1969"
3,D,16168,35.5579,16.7420,"15,471.4946"
4,E,9742,43.0096,19.3643,"18,476.4781"
5,F,3312,50.4227,23.7116,"20,476.1021"
6,G,867,53.2872,26.7509,"20,353.4602"


TIME_TEST - DEFAULT RATE BY GRADE


,grade,n_loans,default_rate,avg_int_rate,avg_loan_amnt
0,A,4111,3.9893,6.8428,"13,789.6132"
1,B,7374,7.3773,10.1487,"12,961.6050"
2,C,8266,12.4365,13.6994,"14,405.3895"
3,D,4433,18.1141,18.0519,"16,090.9147"
4,E,2495,26.9339,21.7277,"18,377.6052"
5,F,1064,31.2030,25.1120,"19,775.1410"
6,G,345,42.0290,28.4758,"19,791.5942"


In [56]:
# Default rate by issue year

vintage_summary = (
    modeling_df
    .assign(issue_year=modeling_df["issue_date"].dt.year)
    .groupby("issue_year")
    .agg(
        n_loans=("default_flag", "size"),
        default_rate=("default_flag", "mean"),
        avg_int_rate=("int_rate", "mean"),
        avg_loan_amnt=("loan_amnt", "mean")
    )
    .reset_index()
)

vintage_summary["default_rate"] = vintage_summary["default_rate"] * 100

display(vintage_summary)

,issue_year,n_loans,default_rate,avg_int_rate,avg_loan_amnt
0,2007,195,16.4103,10.3071,"8,854.4872"
1,2008,1240,15.8065,11.1451,"9,067.8024"
2,2009,3826,12.2582,12.1778,"9,834.1349"
3,2010,9258,13.2102,11.7190,"10,572.0728"
4,2011,17435,15.2452,12.2167,"12,008.8385"
5,2012,41202,16.4798,13.4593,"13,133.8005"
6,2013,97662,15.7400,14.2282,"14,104.7365"
7,2014,102860,23.1110,13.9060,"14,498.7855"
8,2015,94264,24.9013,13.3268,"14,908.1412"
9,2016,28088,13.1302,13.7776,"14,824.6555"


## Split Strategy Decision

The dataset exhibits clear temporal default-rate drift across loan issue years. 
The 2014–2015 vintages show substantially higher realized default rates, while the 2016 vintage shows a lower realized default rate. 
This pattern remains visible even within the same Lending Club grade, suggesting that the shift is not only caused by grade composition.

For the main PD modeling and calibration pipeline, this project uses a stratified random train-validation-test split to maintain stable default-rate distributions and support reliable probability calibration for the pricing simulator.

Calendar-based splits are retained as a robustness diagnostic to evaluate temporal stability and document vintage effects. The model should therefore be interpreted as a portfolio analytics prototype, not as a production-ready vintage forecasting model.

# 6. Save Processed Datasets

In [51]:
modeling_full_path = PROCESSED_DIR / "lending_club_modeling_full.csv"

train_path = PROCESSED_DIR / "train.csv"
valid_path = PROCESSED_DIR / "valid.csv"
test_path = PROCESSED_DIR / "test.csv"

time_train_path = PROCESSED_DIR / "time_train.csv"
time_valid_path = PROCESSED_DIR / "time_valid.csv"
time_test_path = PROCESSED_DIR / "time_test.csv"

modeling_df.to_csv(modeling_full_path, index=False)

# Main development splits
train_df.to_csv(train_path, index=False)
valid_df.to_csv(valid_path, index=False)
test_df.to_csv(test_path, index=False)

# Time-based robustness splits
time_train_df.to_csv(time_train_path, index=False)
time_valid_df.to_csv(time_valid_path, index=False)
time_test_df.to_csv(time_test_path, index=False)

print("Saved files:")
for path in [
    modeling_full_path,
    train_path,
    valid_path,
    test_path,
    time_train_path,
    time_valid_path,
    time_test_path
]:
    print(f"{path} | {path.stat().st_size / (1024 ** 2):,.2f} MB")

Saved files:
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\lending_club_modeling_full.csv | 76.68 MB
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\train.csv | 53.68 MB
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\valid.csv | 11.50 MB
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\test.csv | 11.50 MB
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\time_train.csv | 52.87 MB
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\time_valid.csv | 18.36 MB
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\time_test.csv | 5.45 MB


In [52]:
feature_decision.to_csv(REPORTS_DIR / "02_feature_decision.csv", index=False)
modeling_missing.to_csv(REPORTS_DIR / "02_modeling_missing_summary.csv", index=False)
split_summary.to_csv(REPORTS_DIR / "02_split_summary_main_stratified.csv", index=False)
time_split_summary.to_csv(REPORTS_DIR / "02_split_summary_time_robustness.csv", index=False)
cap_summary.to_csv(REPORTS_DIR / "02_cap_values.csv", index=False)

print("Reports saved to:", REPORTS_DIR)
print("Saved report files:")
print("- 02_feature_decision.csv")
print("- 02_modeling_missing_summary.csv")
print("- 02_split_summary_main_stratified.csv")
print("- 02_split_summary_time_robustness.csv")
print("- 02_cap_values.csv")

Reports saved to: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\reports
Saved report files:
- 02_feature_decision.csv
- 02_modeling_missing_summary.csv
- 02_split_summary_main_stratified.csv
- 02_split_summary_time_robustness.csv
- 02_cap_values.csv


In [53]:
print("Notebook 02 completed.")

print("\nFinal modeling dataframe:")
print(modeling_df.shape)

print("\nMain Stratified Train / Valid / Test:")
print(train_df.shape, valid_df.shape, test_df.shape)

print("\nMain split target rates:")
print("Train:", train_df["default_flag"].mean())
print("Valid:", valid_df["default_flag"].mean())
print("Test :", test_df["default_flag"].mean())

print("\nTime-based robustness Train / Valid / Test:")
print(time_train_df.shape, time_valid_df.shape, time_test_df.shape)

print("\nTime split target rates:")
print("Time Train:", time_train_df["default_flag"].mean())
print("Time Valid:", time_valid_df["default_flag"].mean())
print("Time Test :", time_test_df["default_flag"].mean())

print("\nFeature counts:")
print("Numeric features:", len(primary_numeric_features))
print("Categorical features:", len(primary_categorical_features))

print("\nPrimary numeric features:")
print(primary_numeric_features)

print("\nPrimary categorical features:")
print(primary_categorical_features)

Notebook 02 completed.

Final modeling dataframe:
(396030, 29)

Main Stratified Train / Valid / Test:
(277221, 29) (59404, 29) (59405, 29)

Main split target rates:
Train: 0.19612872040718415
Valid: 0.1961315736314053
Test : 0.19612827203097383

Time-based robustness Train / Valid / Test:
(273678, 29) (94264, 29) (28088, 29)

Time split target rates:
Time Train: 0.18456726518024832
Time Valid: 0.24901340914877365
Time Test : 0.13130162346909713

Feature counts:
Numeric features: 17
Categorical features: 5

Primary numeric features:
['loan_amnt', 'term_months', 'annual_inc_log', 'dti_capped', 'emp_length_num', 'emp_length_missing', 'open_acc', 'pub_rec', 'revol_bal_log', 'revol_util_capped', 'revol_util_missing', 'total_acc', 'mort_acc', 'mort_acc_missing', 'pub_rec_bankruptcies', 'pub_rec_bankruptcies_missing', 'credit_history_length_capped']

Primary categorical features:
['home_ownership', 'verification_status', 'purpose', 'initial_list_status', 'application_type']
